<a href="https://colab.research.google.com/github/HarshaliAlave/hospital-patient-routing-ai/blob/main/hospital_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Installing Libraries

In [ ]:
!pip install -qU langgraph langchain-groq pandas

##Importing the api key

In [ ]:
import os
from getpass import getpass
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")


Enter your Groq API key: ··········


##Data of Doctors

In [ ]:
from google.colab import files
import pandas as pd
uploaded = files.upload()  # select doctors.csv

doctors_df = pd.read_csv("doctors.csv")
doctors_df

Saving doctors.csv to doctors (1).csv


,doctor_name,ward,status
0,Dr. Mehta,general,free
1,Dr. Kapoor,general,busy
2,Dr. Rao,emergency,free
3,Dr. Singh,emergency,busy
4,Dr. Iyer,mental_health,free
5,Dr. Sharma,mental_health,busy


##Creating a shared memory:TypeDict

In [ ]:
from typing import TypedDict, Literal

class PatientState(TypedDict):
    name: str
    age: str
    query: str
    ward: str
    reasoning: str
    assigned_doctor: str

##Creating the intake node

In [ ]:
def intake_node(state: PatientState) -> PatientState:
    print("=== Patient Intake Form ===")
    name = input("Patient name: ")
    age = input("Patient age: ")
    query = input("Describe the problem/symptoms: ")
    return {
        "name": name,
        "age": age,
        "query": query,
        "ward": "",
        "reasoning": "",
        "assigned_doctor": "",
    }

##Creating the agent router node:It will use a LLM to classify the input into the ward categories

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
ROUTER_SYSTEM_PROMPT = """You are a hospital triage router. Based on the patient's
description, classify them into EXACTLY ONE of these wards:
- emergency: life-threatening or urgent physical conditions (severe pain, bleeding,
  breathing difficulty, chest pain, accidents, high fever, unconsciousness, etc.)
- mental_health: psychological/emotional distress (anxiety, depression, panic attacks,
  suicidal thoughts, self-harm, trauma, etc.)
- general: everything else (routine checkups, mild/chronic non-urgent symptoms, colds,
  minor aches, follow-ups, etc.)
Respond with ONLY one word: emergency, mental_health, or general.
"""

# Hard safety-net keywords — these always override the LLM, since misrouting
# a genuine crisis is unacceptable. Good example of "guardrail before AI".
EMERGENCY_KEYWORDS = ["chest pain", "can't breathe", "unconscious", "severe bleeding", "heart attack"]
CRISIS_KEYWORDS = ["suicide", "kill myself", "self-harm", "want to die"]

def router_node(state: PatientState) -> PatientState:
    query_lower = state["query"].lower()
    if any(kw in query_lower for kw in CRISIS_KEYWORDS):
        ward, reasoning = "mental_health", "Crisis keyword detected — routed immediately for safety."
    elif any(kw in query_lower for kw in EMERGENCY_KEYWORDS):
        ward, reasoning = "emergency", "Emergency keyword detected — routed immediately for safety."
    else:
        response = llm.invoke([
            SystemMessage(content=ROUTER_SYSTEM_PROMPT),
            HumanMessage(content=f"Age: {state['age']}\nSymptoms: {state['query']}")
        ])
        raw = response.content.strip().lower()

# Defensive parsing in case the model adds extra words
        if "emergency" in raw:
            ward = "emergency"
        elif "mental_health" in raw or "mental health" in raw:
            ward = "mental_health"
        elif "general" in raw:
            ward = "general"
        else:
            ward = "general"  # safe default
        reasoning = f"LLM classified based on: '{state['query']}'"
    return {**state, "ward": ward, "reasoning": reasoning}

##Defining the wards

In [ ]:
def general_ward_node(state: PatientState) -> PatientState:
    print(f"\n{state['name']} -> GENERAL WARD")
    print(f"Reason: {state['reasoning']}")
    return state

def emergency_ward_node(state: PatientState) -> PatientState:
    print(f"\n{state['name']} -> EMERGENCY WARD (priority)")
    print(f"Reason: {state['reasoning']}")
    return state

def mental_health_ward_node(state: PatientState) -> PatientState:
    print(f"\n{state['name']} -> MENTAL HEALTH WARD")
    print(f"Reason: {state['reasoning']}")
    return state

##Check Doctor availability node

In [ ]:
def doctor_availability_node(state: PatientState) -> PatientState:
    ward = state["ward"]
    available = doctors_df[
       (doctors_df["ward"] == ward) & (doctors_df["status"] == "free")
    ]
    if not available.empty:
        doctor = available.iloc[0]["doctor_name"]
        assigned_doctor = doctor

        # Mark doctor busy for the rest of this Colab session
        doctors_df.loc[doctors_df["doctor_name"] == doctor, "status"] = "busy"
    else:
        assigned_doctor = "No doctor currently free — patient will be queued"
    print(f" Doctor assigned: {assigned_doctor}")
    return {**state, "assigned_doctor": assigned_doctor}

##Building the graph

In [ ]:
from langgraph.graph import StateGraph, END

def route_decision(state: PatientState) -> Literal["general", "emergency", "mental_health"]:
    return state["ward"]

builder = StateGraph(PatientState)

builder.add_node("intake", intake_node)
builder.add_node("router", router_node)
builder.add_node("general", general_ward_node)
builder.add_node("emergency", emergency_ward_node)
builder.add_node("mental_health", mental_health_ward_node)
builder.add_node("doctor_check", doctor_availability_node)
builder.set_entry_point("intake")
builder.add_edge("intake", "router")
builder.add_conditional_edges(
    "router",
    route_decision,
    {
        "general": "general",
        "emergency": "emergency",
        "mental_health": "mental_health",
    },
)
builder.add_edge("general", "doctor_check")
builder.add_edge("emergency", "doctor_check")
builder.add_edge("mental_health", "doctor_check")
builder.add_edge("doctor_check", END)
graph = builder.compile()

##Print graph

In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

##Invoke the graph

In [ ]:
final_state = graph.invoke({})
print("\n=== Final Summary ===")
print(f"Patient: {final_state['name']} (Age: {final_state['age']})")
print(f"Ward: {final_state['ward']}")
print(f"Assigned Doctor: {final_state['assigned_doctor']}")

=== Patient Intake Form ===
Patient name: Harshali Alave
Patient age: 20
Describe the problem/symptoms: Headache and migrain

Harshali Alave -> GENERAL WARD
Reason: LLM classified based on: 'Headache and migrain'
 Doctor assigned: No doctor currently free — patient will be queued

=== Final Summary ===
Patient: Harshali Alave (Age: 20)
Ward: general
Assigned Doctor: No doctor currently free — patient will be queued
